Based on the pigment concentration we observe in eddies, we want to determine how many distinct groups of PFTs we can tell apart via hierarchical clustering. That way, we may make a claim about abundance for a group of PFTs.

We perform this analysis for our target eddies only. Each sample is one eddy-composite: the pigments are first averaged over all interior pixels of a valid eddy-composite, and that mean is one point in the correlations. We average first rather than treating every pixel as a sample, since single-pixel SDP retrievals are much noisier.


1. We divide each of the pigments by Tchla (13 total, with 1 being Tchla, so we have 12 accessory pigments). That way, if two eddies contain the same community composition in terms of PFTs they will have the same pigment ratios regardless of different total biomass.
    - This is NOT normalized.

2. Compute a matrix of Pearson's correlation between every pair of pigment ratios
    - We use 8-day PACE composites. If an eddy is spotted multiple times within that 8-day period it will only count once (otherwise we'd be duplicating the same PACE observations), choosing the contour on a day nearest to the center of the 8-day composite.

3. Use $\sqrt{2(1 - r)}$ as distance metric between two points (0-2 range inclusive). Build with Ward's linkage combination strategy for hierarchical clustering.

4. After determining a cutoff for clusters of pigments, we have groups of pigments that need to be mapped to groups of PFTs. Following Kramer and Siegel (2019), a cluster is named after every PFT whose diagnostic (marker) pigment it contains. Pigments shared by several PFTs (Chlc12, Chlc3, Viola) don't name a cluster; they just sit with whichever markers they correlate with.

Chlorophyll-a is the one pigment that every photosynthetic phytoplankton group carries. However, it's not a direct proxy for biomass since different PFTs can have different amounts of Chl-a per unit of carbon/biomass.

## 1. Imports and settings

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator
from scipy.cluster.hierarchy import cophenet, dendrogram, fcluster, set_link_color_palette

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.utils.pigment_clusters import CLUSTER_CUT_HEIGHT, DIAGNOSTIC_PIGMENTS, GROUP_PIGMENTS, PIGMENT_GROUPS, RATIO_PIGMENTS, SDP_PIGMENTS, build_ward_linkage, correlate_ratios, label_clusters, load_target_composites

EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
from eddy_tracking.config import load_analysis_config
cfg = load_analysis_config(EXPERIMENT)
pigments = list(SDP_PIGMENTS)
accessory = list(RATIO_PIGMENTS)
pigment_labels = {
    'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'ButFuco': "19'-But-fucoxanthin",
    'Perid': 'Peridinin', 'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin',
    'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
}

## 2. Pigment-to-group map

From Table 1 of Kramer and Siegel (2019): every group in which a pigment is often present, always present, or unique.

- Diatoms: Fuco, Chlc12, Chlc3
- Dinoflagellates: Fuco, Perid, Chlc12, Chlc3
- Chrysophytes: Fuco, Chlc12, Viola, Zea
- Pelagophytes: Fuco, ButFuco, Chlc12, Zea
- Haptophytes: Fuco, HexFuco, ButFuco, Chlc12, Chlc3
- Cryptophytes: Chlc12, Allo
- Green algae: MV_chlb, Neo, Viola, Zea
- Cyanobacteria: Zea, DV_chla

Clusters are labeled by diagnostic pigments only. Kramer and Siegel (2019) name each branch of their trees by its marker pigments: Fuco for diatoms, Perid for dinoflagellates, HexFuco for haptophytes, MVchlb with the other green algal carotenoids for green algae, and DVchla with Zea for cyanobacteria. Their local trees add Allo for cryptophytes and ButFuco for the ButFuco-bearing flagellates. The same markers define the diagnostic pigment analysis of Vidussi et al. (2001) and Uitz et al. (2006).

| Diagnostic pigment | PFT |
| --- | --- |
| Fuco | Diatoms |
| Perid | Dinoflagellates |
| ButFuco | Pelagophytes |
| HexFuco | Haptophytes |
| Allo | Cryptophytes |
| MV_chlb, Neo | Green algae |
| Zea, DV_chla | Cyanobacteria |

Chrysophytes have no diagnostic pigment among the 12 SDP pigments, so no cluster can be named after them. Chlc12, Chlc3, and Viola occur in several PFTs and never name a cluster.

These lists cover the 12 accessory pigments included in this analysis. Both tables, the target eddy-composites, the ratio tree, the cut height, and the labeling rule live in `eddy_tracking.utils.pigment_clusters`, which `figures.ipynb` also reads.

In [ ]:
pigment_groups = PIGMENT_GROUPS
# For each group, list the pigments that it contains
group_pigments = GROUP_PIGMENTS
# The marker pigment of each PFT, the only pigments that name a cluster
diagnostic_pigments = DIAGNOSTIC_PIGMENTS


## 3. Target eddies and their composites

In [ ]:
# One row per target eddy-composite, with the interior-mean pigments: cyclones that crossed the axis north to south, anticyclones south to north
target_pigments = load_target_composites(DATA_DIR, cfg)


## 4. Figure style and sample counts

Checkpoint: the number of target eddies, eddy-composites per polarity.

In [ ]:
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

display(cast(pd.DataFrame, target_pigments.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
)))

## 5. Correlation matrix

The 12 x 12 correlation matrix of the pigment-to-Tchla ratios.

In [ ]:
# Each eddy-composite is one data point: every pigment is the mean over the composite's interior pixels, divided by the interior-mean Tchla
correlation = correlate_ratios(target_pigments)

## 6. Cluster the pigments and draw the tree

The cell builds the tree, records the fit and the cluster numbers, and draws the dendrogram.

Below the diagram, every PFT is listed with the diagnostic pigments each cluster holds for it. A PFT is bold when some cluster holds one of its markers; Chrysophytes stay blank because they have no marker among the SDP pigments.

In [ ]:
def draw_tree(corr: np.ndarray, n: int, cutoff: float) -> tuple[dict, np.ndarray]:
    set_link_color_palette(['#1b9e77', '#d95f02', '#7570b3', '#e7298a'])
    distance, Z = build_ward_linkage(corr)
    # Cophenetic r: how well the tree heights reproduce the distances
    fit = {'n': n, 'cophenetic_r': cophenet(Z, distance)[0]}
    cluster_numbers = fcluster(Z, t=cutoff, criterion='distance')
    cluster_fig, tree_ax = cast('tuple[Figure, Axes]', plt.subplots(figsize=(6.69, 4.9)))
    cluster_fig.subplots_adjust(left=0.075, right=0.99, bottom=0.29, top=0.98)
    tree = dendrogram(Z, ax=tree_ax, labels=[pigment_labels[pigment] for pigment in accessory], leaf_rotation=90, leaf_font_size=8, color_threshold=cutoff, above_threshold_color='#444444')
    for label, color in zip(tree_ax.xaxis.get_ticklabels(), tree['leaves_color_list']):
        label.set_color(color)
    # Label each cluster with the groups whose diagnostic pigments sit in it
    cluster_labels = {cluster.cluster_id: cluster.labels for cluster in label_clusters(pd.Series(cluster_numbers, index=accessory))}
    tree_labels = []
    for cluster_id in np.unique(cluster_numbers):
        # By cluster number, not leaf color: every single-pigment cluster takes the same above-cutoff gray
        members = [position for position, leaf in enumerate(tree['leaves']) if cluster_numbers[leaf] == cluster_id]
        # Leaves sit at x = 5, 15, 25, ..., so 10 * mean position + 5 is the middle of the cluster
        tree_labels.append(tree_ax.text(10 * float(np.mean(members)) + 5, 0.98, ',\n'.join(cluster_labels[cluster_id]), transform=tree_ax.get_xaxis_transform(), ha='center', va='top', color=tree['leaves_color_list'][members[0]]))
    tree_ax.axhline(cutoff, color='#999999', linewidth=0.6, linestyle=(0, (4, 2.5)))  # pyright: ignore[reportArgumentType]
    tree_ax.tick_params(axis='x', length=0)
    tree_ax.spines['bottom'].set_visible(False)
    pft_rows = []
    for group in group_pigments:
        # The diagnostic pigments of this PFT in each cluster, blank when the cluster holds none
        markers = {
            f'Cluster {cluster_id}': ', '.join(
                pigment
                for pigment, assigned_cluster in zip(accessory, cluster_numbers)
                if assigned_cluster == cluster_id and diagnostic_pigments.get(pigment) == group
            )
            for cluster_id in np.unique(cluster_numbers)
        }
        pft_rows.append({'PFT': group, **markers})
    tree_ax.autoscale(axis='y')
    ticks = cast(np.ndarray, MaxNLocator(nbins=4, steps=[1, 2, 5, 10]).tick_values(0, tree_ax.get_ylim()[1]))
    ticks = ticks[ticks <= tree_ax.get_ylim()[1]]
    # Extra room above the tree for the group labels
    tree_ax.set_ylim(0, 1.5 * tree_ax.get_ylim()[1])
    tree_ax.yaxis.set_ticks(ticks)
    # Keep the left spine from running into the labels
    tree_ax.spines['left'].set_bounds(0, ticks[-1])
    tree_ax.set_ylabel('Linkage distance')
    # Keep every label on one line: widen the figure in 5% steps until each label clears its neighbors by a few points
    renderer = cluster_fig.canvas.get_renderer()  # pyright: ignore[reportAttributeAccessIssue]
    for _ in range(40):
        boxes = sorted((label.get_window_extent(renderer) for label in tree_labels if label.get_text()), key=lambda box: box.x0)
        if all(right.x0 >= left.x1 + 0.1 * cluster_fig.dpi for left, right in zip(boxes, boxes[1:])):
            break
        cluster_fig.set_size_inches(1.05 * cluster_fig.get_figwidth(), cluster_fig.get_figheight())
    plt.show()
    pft_table = pd.DataFrame(pft_rows).set_index('PFT')
    # Resolved: the cluster holds at least one of the PFT's diagnostic pigments
    is_resolved = pft_table.map(lambda markers: markers != '')
    display(pft_table.style
        .apply(lambda _: np.where(is_resolved, 'font-weight: bold', ''), axis=None)
        .apply_index(lambda _: np.where(is_resolved.any(axis=1), 'font-weight: bold', '')))
    return fit, cluster_numbers


# CLUSTER_CUT_HEIGHT sits inside the gap between the merges at 0.68 and 0.76 of the ratio tree, the six-cluster level
fit, cluster_numbers = draw_tree(correlation.to_numpy(), len(target_pigments), cutoff=CLUSTER_CUT_HEIGHT)
# Cophenetic r checks how well the tree heights reproduce the pigment distances, the test Kramer and Siegel (2019) run before cutting
display(pd.DataFrame([fit]).round(3))
# The labeled clusters that figures.ipynb groups its pigment panels by
display(pd.DataFrame([
    {'cluster': cluster.cluster_id, 'PFTs': cluster.label, 'pigments': ', '.join(cluster.pigments)}
    for cluster in label_clusters(pd.Series(cluster_numbers, index=accessory))
]))

### Choosing the cut height

Kramer and Siegel (2019) cut each tree at the linkage distance that gives "a maximum number of taxonomically relevant clusters." Read from the top of the tree with the diagnostic-pigment labels:

- 3 clusters (cut 1.45 to 2.60): diatoms; dinoflagellates, pelagophytes, haptophytes, and cyanobacteria; cryptophytes and green algae.
- 4 clusters (0.99 to 1.45): Chlc3 splits off alone. It carries no marker, so no new group.
- 5 clusters (0.76 to 0.99): Perid splits off, giving dinoflagellates their own cluster.
- 6 clusters (0.68 to 0.76): Zea and DV_chla split from HexFuco and ButFuco, separating cyanobacteria from haptophytes and pelagophytes.
- 7 clusters and below: Neo splits from the green algae, then Fuco from Chlc12. These repeat groups that already have a cluster.

The six-cluster level is the lowest cut where every split still separates a new group, so `CLUSTER_CUT_HEIGHT` is 0.72. Its window (0.68 to 0.76) is much narrower than the three-cluster window (1.45 to 2.60), so the cell below resamples whole eddies to check how often the two key splits survive. The three-cluster cut serves as the robustness check.

In [ ]:
# Resample whole eddies with replacement, rebuild the ratio tree, cut it into the same number of clusters, and count how often each split holds
N_RESAMPLES = 500
rng = np.random.default_rng(2026)
eddy_keys = target_pigments[['polarity', 'track_id']].drop_duplicates()
by_eddy = {key: rows for key, rows in target_pigments.groupby(['polarity', 'track_id'])}
n_main = len(np.unique(cluster_numbers))
split_rows = []
for n_clusters in (3, n_main):
    agreement = []
    cyanobacteria_apart = perid_alone = 0
    full = fcluster(build_ward_linkage(correlation.to_numpy())[1], t=n_clusters, criterion='maxclust')
    upper = np.triu_indices(len(accessory), 1)
    for _ in range(N_RESAMPLES):
        sample = pd.concat([by_eddy[tuple(key)] for key in eddy_keys.to_numpy()[rng.integers(0, len(eddy_keys), len(eddy_keys))]])
        resampled = fcluster(build_ward_linkage(correlate_ratios(sample).to_numpy())[1], t=n_clusters, criterion='maxclust')
        # Share of pigment pairs that are together (or apart) in both the resampled and the full tree
        agreement.append(np.mean((full[:, None] == full[None, :])[upper] == (resampled[:, None] == resampled[None, :])[upper]))
        cluster_of = dict(zip(accessory, resampled))
        cyanobacteria_apart += cluster_of['Zea'] == cluster_of['DV_chla'] and cluster_of['HexFuco'] == cluster_of['ButFuco'] and cluster_of['Zea'] != cluster_of['HexFuco']
        perid_alone += cluster_of['Perid'] not in (cluster_of['HexFuco'], cluster_of['Zea'], cluster_of['Fuco'])
    split_rows.append({
        'clusters': n_clusters, 'median pair agreement': np.median(agreement), '5th percentile': np.percentile(agreement, 5),
        'cyanobacteria apart from haptophytes': cyanobacteria_apart / N_RESAMPLES, 'Perid on its own': perid_alone / N_RESAMPLES,
    })
display(pd.DataFrame(split_rows).round(2))

The grouping of alloxanthin with monovinyl chlorophyll-b, neoxanthin, and violaxanthin matches observations of correlated pigment-to-total-chlorophyll-a ratios in the western North Atlantic ([Kramer et al., 2020](https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2020.00215/full)). This association also appears globally, although its strength varies by region ([Kramer & Siegel, 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC7043335/)). It supports cryptophyte and green-algal pigment covariation, which is why this cluster carries both labels; it does not establish a universal correlation between their abundances.

Investigate why Haptophytes and Cyanobacteria are not separating out when Dr. Kramer's analysis had them separated cleanly. Is this due to mixing up seasons and water masses? What if we group by water mass + season? How does this compare to the whole background?